In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Latent-Landmark-Sync-V1 · M1 clean 开发
本 notebook 本轮仅作静态交付，未执行。M0 CPU 合成检查不证明真实载体存活。固定 4 来源 × 6 臂 = 24 幅最终 RGB；全部失败与未执行保留。仅 M1 clean，不执行 M2/M3。
需要用户自行连接可用运行时，并在 Colab Secrets 设置 `HF_TOKEN` 和既有 `CEG_WM_ROOT_KEY`。使用 main 内容 ISS、参考阈值和 SyncSeal 0.75；不继承原 FPR。
默认 `MODE='run'`，用户运行全部单元即可执行完整 M1；可改为 `initialize` 仅建立 24 行未执行名单，或 `summarize` 汇总已有输出。同一输出不自动重试。

In [ ]:
MODE = 'run' # @param ['run', 'initialize', 'summarize']
SOURCE_EXACT = '837ffe313a0de7cb2287676a4b6071913242d1cc'
from pathlib import Path
REPO = Path('/content/ceg-wm-latent-landmark-' + SOURCE_EXACT[:12])
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/latent-landmark-sync-v1-m1')
RUNTIME_ROOT = Path('/content/latent-landmark-sync-v1-runtime')
print({'mode': MODE, 'source_exact': SOURCE_EXACT, 'output': str(OUTPUT), 'planned_images': 24, 'independent_sources': 4})

## 1. 固定实现与依赖
复用历史成功入口的依赖范围；实际版本与模型配置由运行记录保留。Notebook 固定的源码提交与承载本 Notebook 的提交可以不同，避免自引用。

In [ ]:
import subprocess, sys, os
assert len(SOURCE_EXACT) == 40 and all(c in '0123456789abcdef' for c in SOURCE_EXACT)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'dev/latent-landmark-sync-v1', '--single-branch', 'https://github.com/RICHAAARC/CEG-WM.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_EXACT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_EXACT
dependencies = ['diffusers<0.40', 'transformers', 'accelerate', 'lpips', 'torchmetrics', 'matplotlib']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *dependencies, str(REPO)], check=True)
if MODE == 'run':
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
print({'code': SOURCE_EXACT, 'mode': MODE})

## 2. 执行所选模式
`run` 加载真实模型：step 18 实际 ISS callback 后写入；标准 step-end callback 轨迹验证剩余一次更新；最终 RGB8 确定性 VAE mode 重新编码后盲读。未阈值峰先落盘，随后固定共享负对照阈值并评价。

In [ ]:
command = [sys.executable, '-m', 'experiments.run_latent_landmark_sync_v1', '--mode', MODE, '--output', str(OUTPUT), '--runtime-root', str(RUNTIME_ROOT)]
subprocess.run(command, cwd=REPO, check=True)
print('结果目录:', OUTPUT)

## 3. 结果与边界
`rows.json` 保存全部 24 行、最终图路径、盲峰、内容和质量；`evaluated.json` 保留逐点误差；`summary.json` 保留所有幅度和四来源分母。无完整真实结果时只标记未执行/不完整。开发无可读窗口则停止首候选，不扩大参数。

In [ ]:
import json
summary_path = OUTPUT / 'summary.json'
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))
else:
    print('NOT_EXECUTED: 尚无 summary.json')